# Data Science for Business - Stacking on Micro Mortgages

*Session 7 · Not covered in ISLP; builds on ISLP 4.3 (logistic regression), 5.1.3 (k-fold CV), 8.2.1 (averaging reduces variance) and the L1 logistic regression in Lab 10.9.5*

Stacking goes beyond ISLP. For further reading, see Hastie, Tibshirani & Friedman, *The Elements of Statistical Learning*, Section 8.8 "Model Averaging and Stacking".

## Initialize notebook
Load required packages. Set up workspace, e.g., set theme for plotting and initialize the random number generator.

In [ ]:
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_predict
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, StackingClassifier

In [ ]:
np.random.seed(42)

# scikit-learn >= 1.8 announces that `penalty='l1'` will be replaced by `l1_ratio=1`. We keep penalty='l1',
# because it is the spelling that also gives a lasso on older versions (e.g., on Colab), and silence only these two notices.
warnings.filterwarnings('ignore', message="'penalty' was deprecated", category=FutureWarning)
warnings.filterwarnings('ignore', message="Inconsistent values: penalty=", category=UserWarning)

## Problem description

In India, there are about 20 million home loan (mortgage) aspirants
working in the informal sector:

- Monthly income between INR 20,000-25,000 (\$ 325-400)
- Typically no formal accounts and documents (e.g., tax returns, income proofs, bank statements)
- Often use services of money lenders with interest rates between 30 and 60% per annum

Providing mortgages to this group of customers requires to quickly and
efficiently assess their creditworthiness. Due to a lack of formal
documents and objective data, most financial institutions perform
interview-based processes to decide about these loan requests:

Strength of the current process:

-   Interview-based field assessment

-   Relaxation of document requirements

Weaknesses of the current process:

-   Costly (total transaction costs as high as 30% of loan volume)

-   Subjective judgments; depends on individual skills and motivations

-   Low reliability across branches and credit officers

-   Risk of corruption and fraud

## Load data

Load training data from CSV file.

In [ ]:
data = pd.read_csv('https://raw.githubusercontent.com/olivermueller/ds4b-2026/main/Session_07/micromortgage.csv')

## Prepare data

We use the same preparation steps as in the tree-based models notebook: drop the ID, make `Tier` categorical, do a stratified train/test split (80% of the requests are approved), one-hot encode the categorical and standardize the numerical predictors.

In [ ]:
data = data.drop(['ID'], axis=1)
data["Tier"] = data["Tier"].apply(lambda x: "T"+str(x))

In [ ]:
X = data.drop("Decision", axis=1)
y = data["Decision"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
categorical_features = X_train.select_dtypes(include=['object', 'string']).columns
numerical_features = X_train.select_dtypes(exclude=['object', 'string']).columns

In [ ]:
enc = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
enc.fit(X_train[categorical_features])

X_train_cat = enc.transform(X_train[categorical_features])
X_test_cat = enc.transform(X_test[categorical_features])

X_train_cat = pd.DataFrame(X_train_cat, columns=enc.get_feature_names_out(categorical_features))
X_test_cat = pd.DataFrame(X_test_cat, columns=enc.get_feature_names_out(categorical_features))

In [ ]:
scaler = StandardScaler()
scaler.fit(X_train[numerical_features])

X_train_num = scaler.transform(X_train[numerical_features])
X_test_num = scaler.transform(X_test[numerical_features])

X_train_num = pd.DataFrame(X_train_num, columns=numerical_features)
X_test_num = pd.DataFrame(X_test_num, columns=numerical_features)

In [ ]:
X_train = pd.concat([X_train_num, X_train_cat], axis=1)
X_test = pd.concat([X_test_num, X_test_cat], axis=1)

## Base model 1: lasso

Our first base model is a lasso logistic regression. scikit-learn's `LogisticRegression` uses an L2 (ridge) penalty by default; `penalty='l1'` (with the `liblinear` solver) makes it a lasso. `C` is the inverse of λ (ISLP Lab 10.9.5): a small C means strong shrinkage. We tune C with 5-fold cross-validation and use the AUC as the criterion.

In [ ]:
model_lasso_tuned = GridSearchCV(
                estimator=LogisticRegression(penalty='l1', solver='liblinear', max_iter=10000, random_state=42),
                param_grid={
                        'C': np.logspace(-4, 4, 10)
                    }, scoring='roc_auc', cv=5, n_jobs=-1
                )

In [ ]:
model_lasso_tuned.fit(X_train, y_train)

In [ ]:
model_lasso_tuned.best_estimator_

Like in ISLP 6.2.2, the lasso sets some coefficients exactly to zero. How many predictors does the tuned model keep?

In [ ]:
print('Non-zero coefficients:', (model_lasso_tuned.best_estimator_.coef_ != 0).sum(), 'of', X_train.shape[1])

Make predictions on the test set and evaluate the model.

In [ ]:
pred_lasso_label = model_lasso_tuned.predict(X_test)
pred_lasso_proba = model_lasso_tuned.predict_proba(X_test)[:,1]

In [ ]:
print(classification_report(y_test, pred_lasso_label))

In [ ]:
roc_auc_score(y_test, pred_lasso_proba)

## Base model 2: random forest

Our second base model is the random forest from the tree-based models notebook (500 trees, m tuned via `max_features`).

In [ ]:
model_rf_tuned = GridSearchCV(
                estimator=RandomForestClassifier(n_estimators=500, random_state=42),
                param_grid={
                        'max_features': ['sqrt', 0.33, None]
                    }, scoring='roc_auc', cv=5, n_jobs=-1
                )

In [ ]:
model_rf_tuned.fit(X_train, y_train)

In [ ]:
model_rf_tuned.best_estimator_

In [ ]:
pred_rf_label = model_rf_tuned.predict(X_test)
pred_rf_proba = model_rf_tuned.predict_proba(X_test)[:,1]

In [ ]:
print(classification_report(y_test, pred_rf_label))

In [ ]:
roc_auc_score(y_test, pred_rf_proba)

## Combine models by averaging (blending)

The simplest way to combine models is to average their predicted probabilities with fixed (here: equal) weights. This is often called **blending** or model averaging. It is not yet stacking, because nothing is learned about how to weight the models.

Averaging helps most when the models make different errors (a linear model vs. trees). Like bagging, averaging reduces variance (ISLP 8.2.1).

In [ ]:
pred_avg_proba = (pred_lasso_proba + pred_rf_proba) / 2

In [ ]:
roc_auc_score(y_test, pred_avg_proba)

## Stacking with a meta-learner

In **stacking**, a second-level model, the **meta-learner**, learns how much to trust each base model: its inputs (predictors) are the base models' predicted probabilities, its coefficients are the weights. We use a plain (unpenalized) logistic regression (ISLP 4.3) as the meta-learner.

To train the meta-learner, we need base-model predictions for the training data. But should we simply use the predictions of the base models on the data they were trained on? Let's compare the random forest's AUC on the training set with its cross-validated AUC.

In [ ]:
print('RF AUC on training set (in-sample):', round(roc_auc_score(y_train, model_rf_tuned.predict_proba(X_train)[:,1]), 2))
print('RF AUC in cross-validation:        ', round(model_rf_tuned.best_score_, 2))

The random forest has almost memorised the training set. A meta-learner trained on these in-sample predictions would believe that the forest is (nearly) perfect and give it far too much weight.

The solution are **out-of-fold predictions**: with 5-fold cross-validation (ISLP 5.1.3), each training observation is predicted by a model that was fit on the other four folds, i.e., without seeing this observation. `cross_val_predict` does exactly this.

In [ ]:
yhat_1_train = cross_val_predict(model_lasso_tuned.best_estimator_, X_train, y_train, cv=5, method='predict_proba')[:, 1]
yhat_2_train = cross_val_predict(model_rf_tuned.best_estimator_, X_train, y_train, cv=5, method='predict_proba')[:, 1]

These out-of-fold predictions become the predictors of the meta-learner; the response is the original target.

In [ ]:
stacked_predictions_train = pd.DataFrame({'Lasso': yhat_1_train, 'RandomForest': yhat_2_train})
stacked_predictions_train['target'] = y_train.values
stacked_predictions_train

Fit the meta-learner.

In [ ]:
# C = 1/λ; a huge C means practically no penalty (as in ISLP Lab 4.7.6)
model_stacked = LogisticRegression(C=1e10)
model_stacked.fit(stacked_predictions_train[['Lasso', 'RandomForest']], stacked_predictions_train['target'])

The coefficients tell us how much the meta-learner trusts each base model.

In [ ]:
pd.Series(model_stacked.coef_[0], index=['Lasso', 'RandomForest'])

To make predictions for the test set, we first let the base models (trained on the full training set) predict the test observations, and then feed these predictions into the meta-learner.

In [ ]:
yhat_1_test = model_lasso_tuned.predict_proba(X_test)[:,1]
yhat_2_test = model_rf_tuned.predict_proba(X_test)[:,1]
stacked_predictions_test = pd.DataFrame({'Lasso': yhat_1_test, 'RandomForest': yhat_2_test})

yhat_stacked = model_stacked.predict_proba(stacked_predictions_test[['Lasso', 'RandomForest']])[:,1]
stacked_predictions_test["Meta"] = yhat_stacked
stacked_predictions_test

In [ ]:
roc_auc_score(y_test, yhat_stacked)

Compare the test AUC of the stacked model with the two base models and the simple average. With only two base models and about 1,000 training observations, the gains from stacking are typically small; the test set has only 51 rejections, so small differences in AUC are within noise.

### The same in one step: `StackingClassifier`

scikit-learn bundles all of the above steps (out-of-fold predictions via `cv=5`, fitting the meta-learner, refitting the base models on the full training set) in `StackingClassifier`. Since it uses the same folds as `cross_val_predict`, the meta-learner's coefficients and the test AUC are the same as ours (up to numerical precision).

In [ ]:
model_stacking = StackingClassifier(
                estimators=[('lasso', model_lasso_tuned.best_estimator_), ('rf', model_rf_tuned.best_estimator_)],
                final_estimator=LogisticRegression(C=1e10),
                cv=5, stack_method='predict_proba', n_jobs=-1
                )
model_stacking.fit(X_train, y_train)

In [ ]:
print('Meta-learner coefficients:', model_stacking.final_estimator_.coef_)
print('AUC on test set:', roc_auc_score(y_test, model_stacking.predict_proba(X_test)[:,1]))

## Your turn!

**Your task:** Use a different meta-learner (e.g., decision tree, random forest, gradient boosting machine) for stacking, trained on the out-of-fold predictions `stacked_predictions_train`. Evaluate the performance of your stacked model on the test set and compare it with the stacked model with logistic regression as meta-learner.

**Bonus:** Replace the random forest with a boosted trees model (`HistGradientBoostingClassifier`) as the second base model and compare the results.

In [ ]:
# YOUR CODE HERE